# 02 — Preprocessing & Feature Engineering
**Loan Default Prediction Project**

Goal: build a leak-proof preprocessing pipeline, engineer 6 new features, and check multicollinearity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import warnings, sys, os

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))
from src.features import add_features, get_feature_names

RANDOM_STATE = 42
PLOTS_DIR = '../plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/loan_default_dataset.csv')
X = df.drop(columns=['default'])
y = df['default']
print(f'X shape: {X.shape}  |  y shape: {y.shape}')
print(f'Default rate: {y.mean():.4f}')

## 2. Train / Test Split (Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape}   Test: {X_test.shape}')
print(f'Train default rate: {y_train.mean():.4f}')
print(f'Test  default rate: {y_test.mean():.4f}')
# Both should be ~0.1862 — confirms stratification worked

## 3. Feature Engineering
> All transformations use `add_features()` from `src/features.py` — the same function is imported by `app.py` and `predict.py`, guaranteeing consistency.

In [ ]:
X_train_fe = add_features(X_train)
X_test_fe  = add_features(X_test)

print('New features added:')
new_cols = [c for c in X_train_fe.columns if c not in X_train.columns]
for col in new_cols:
    print(f'  + {col}')
print(f'\nTotal feature count: {X_train_fe.shape[1]}')

In [ ]:
# Check engineered feature distributions
X_train_fe[new_cols].describe().T.round(3)

In [ ]:
# Correlation of engineered features with target
from scipy.stats import pointbiserialr

train_with_target = X_train_fe.copy()
train_with_target['default'] = y_train.values

print('Correlation of all features with target:')
print('-'*55)
for col in X_train_fe.columns:
    valid = train_with_target[[col,'default']].dropna()
    r, p  = pointbiserialr(valid[col], valid['default'])
    stars = '***' if p<0.001 else ''
    print(f'{col:30s}  r={r:+.4f}  {stars}')

## 4. Build Preprocessing Pipeline

In [ ]:
all_features = X_train_fe.columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, all_features),
])

# Fit ONLY on training data
X_train_proc = preprocessor.fit_transform(X_train_fe)
X_test_proc  = preprocessor.transform(X_test_fe)

print(f'Processed train shape: {X_train_proc.shape}')
print(f'Processed test  shape: {X_test_proc.shape}')

In [ ]:
# Sanity check: mean ≈ 0, std ≈ 1 for all columns
means = X_train_proc.mean(axis=0).round(3)
stds  = X_train_proc.std(axis=0).round(3)
print('Post-scaling sanity check (should be ~0 mean, ~1 std):')
for name, m, s in zip(all_features, means, stds):
    print(f'{name:30s}  mean={m:+.3f}  std={s:.3f}')

## 5. Multicollinearity Check (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_df = pd.DataFrame({
    'feature': all_features,
    'VIF': [variance_inflation_factor(X_train_proc, i) for i in range(X_train_proc.shape[1])]
}).sort_values('VIF', ascending=False)

print('VIF Analysis (>10 = high multicollinearity):')
print(vif_df.to_string(index=False))

# Plot VIF
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#993C1D' if v > 10 else '#185FA5' for v in vif_df['VIF']]
ax.barh(vif_df['feature'], vif_df['VIF'], color=colors)
ax.axvline(10, color='red', linestyle='--', linewidth=1, label='VIF=10 threshold')
ax.set_xlabel('Variance Inflation Factor')
ax.set_title('VIF — Multicollinearity Check', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/05_vif_check.png')
plt.show()

## 6. Save Processed Data for Modeling

In [ ]:
import joblib

# Save train/test splits with engineered features (pre-scaling) for the modeling notebook
joblib.dump((X_train_fe, X_test_fe, y_train, y_test, preprocessor, all_features),
            '../models/preprocessing_artifacts.pkl')

print('Saved: models/preprocessing_artifacts.pkl')
print('\nContents: X_train_fe, X_test_fe, y_train, y_test, preprocessor, all_features')

## Summary

- Train set: 240,000 rows · Test set: 60,000 rows (stratified)
- 9 original features + 6 engineered = **15 features total**
- Preprocessing: median imputation → StandardScaler (fitted on train only)
- No VIF > 10 — multicollinearity is acceptable
- All artifacts saved to `models/` for notebook 03